In [25]:
import os
import random
import requests
import urllib3
import numpy as np
import geopandas as gpd
import rasterio
from rasterio import features
from rasterio.windows import Window
from tqdm import tqdm

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

print("All libraries imported successfully.")

All libraries imported successfully.


In [26]:
# 1. Setup Directories
RAW_IMG_DIR = "../01_data/raw/swissimage_production"
os.makedirs(RAW_IMG_DIR, exist_ok=True)

# 2. Get the geographic bounding box
print("Loading Zurich Biotope bounds...")
gdf = gpd.read_file("../01_data/processed/zuerich_biotopes_superclasses.gpkg")
gdf_wgs84 = gdf.to_crs(epsg=4326)
minx, miny, maxx, maxy = gdf_wgs84.total_bounds
print(f"Zurich Bounding Box: {minx:.4f}, {miny:.4f}, {maxx:.4f}, {maxy:.4f}")

# 3. Query the Swisstopo API
print("\nQuerying Swisstopo API for intersecting 10cm tiles...")
STAC_API_URL = "https://data.geo.admin.ch/api/stac/v0.9/collections/ch.swisstopo.swissimage-dop10/items"

params = {
    "bbox": f"{minx},{miny},{maxx},{maxy}",
    "limit": 150
}

response = requests.get(STAC_API_URL, params=params, verify=False)
response.raise_for_status()
items = response.json().get("features", [])
print(f"Found {len(items)} SWISSIMAGE tiles covering Zurich.")

def download_file(url, dest_path):
    if os.path.exists(dest_path):
        return
        
    resp = requests.get(url, stream=True, verify=False)
    resp.raise_for_status()
    total_size = int(resp.headers.get('content-length', 0))
    
    with open(dest_path, 'wb') as file, tqdm(
        desc=os.path.basename(dest_path),
        total=total_size,
        unit='iB',
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for data in resp.iter_content(chunk_size=1024*1024):
            size = file.write(data)
            bar.update(size)

MAX_DOWNLOADS = 150

print(f"\nStarting download of {MAX_DOWNLOADS} tiles...")
downloaded_count = 0

for item in items:
    if downloaded_count >= MAX_DOWNLOADS:
        break
        
    assets = item.get("assets", {})
    for asset_key, asset_info in assets.items():
        # Specifically target the 10cm resolution TIF files
        if "_0.1_" in asset_key and asset_key.endswith((".tif", ".tiff")):
            download_url = asset_info["href"]
            file_name = download_url.split("/")[-1]
            dest_path = os.path.join(RAW_IMG_DIR, file_name)
            
            download_file(download_url, dest_path)
            downloaded_count += 1
            break

print("\n✅ Download phase complete!")

Loading Zurich Biotope bounds...
Zurich Bounding Box: 8.4480, 47.3202, 8.6255, 47.4347

Querying Swisstopo API for intersecting 10cm tiles...
Found 100 SWISSIMAGE tiles covering Zurich.

Starting download of 150 tiles...


swissimage-dop10_2019_2676-1246_0.1_2056.tif: 100%|█| 60.0M/60.0M [00:01<00:00, 
swissimage-dop10_2019_2676-1247_0.1_2056.tif: 100%|█| 62.1M/62.1M [00:01<00:00, 
swissimage-dop10_2019_2676-1248_0.1_2056.tif: 100%|█| 64.4M/64.4M [00:01<00:00, 
swissimage-dop10_2019_2676-1249_0.1_2056.tif: 100%|█| 60.6M/60.6M [00:01<00:00, 
swissimage-dop10_2019_2676-1250_0.1_2056.tif: 100%|█| 55.9M/55.9M [00:01<00:00, 
swissimage-dop10_2019_2676-1251_0.1_2056.tif: 100%|█| 57.4M/57.4M [00:01<00:00, 
swissimage-dop10_2019_2676-1252_0.1_2056.tif: 100%|█| 64.2M/64.2M [00:01<00:00, 
swissimage-dop10_2019_2676-1253_0.1_2056.tif: 100%|█| 55.7M/55.7M [00:01<00:00, 
swissimage-dop10_2019_2676-1254_0.1_2056.tif: 100%|█| 51.0M/51.0M [00:01<00:00, 
swissimage-dop10_2019_2677-1241_0.1_2056.tif: 100%|█| 56.6M/56.6M [00:01<00:00, 
swissimage-dop10_2019_2677-1242_0.1_2056.tif: 100%|█| 54.1M/54.1M [00:01<00:00, 
swissimage-dop10_2019_2677-1243_0.1_2056.tif: 100%|█| 53.1M/53.1M [00:01<00:00, 
swissimage-dop10_2019_2677-1


✅ Download phase complete!


In [27]:
BASE_OUT = "../01_data/model_ready_production"
splits = ['train', 'val', 'test']
for split in splits:
    os.makedirs(f"{BASE_OUT}/{split}/images", exist_ok=True)
    os.makedirs(f"{BASE_OUT}/{split}/masks", exist_ok=True)

all_tifs = sorted([f for f in os.listdir(RAW_IMG_DIR) if f.endswith(('.tif', '.tiff'))])
random.seed(42)
random.shuffle(all_tifs)

num_train = int(len(all_tifs) * 0.8)
num_val = int(len(all_tifs) * 0.1)

train_tifs = all_tifs[:num_train]
val_tifs = all_tifs[num_train:num_train+num_val]
test_tifs = all_tifs[num_train+num_val:]

def get_split(filename):
    if filename in train_tifs: return "train"
    elif filename in val_tifs: return "val"
    else: return "test"

PATCH_SIZE = 1024 
print(f"Processing {len(all_tifs)} geographic tiles into {PATCH_SIZE}x{PATCH_SIZE} patches...")

gdf_projected = gpd.read_file("../01_data/processed/zuerich_biotopes_superclasses.gpkg")

possible_cols = ['label_id', 'superclass_id', 'superclass', 'class_id', 'id']
label_col = None
for col in possible_cols:
    if col in gdf_projected.columns:
        label_col = col
        break

if label_col is None:
    print("Available columns:", gdf_projected.columns.tolist())
    raise ValueError("Could not find the target class column. Please update 'possible_cols' list.")
else:
    print(f"✅ Found label column: '{label_col}'")

patch_counts = {'train': 0, 'val': 0, 'test': 0}

for tif_name in tqdm(all_tifs, desc="Processing Tiles"):
    split = get_split(tif_name)
    img_path = os.path.join(RAW_IMG_DIR, tif_name)
    
    with rasterio.open(img_path) as src:
        bbox = src.bounds
        intersecting_gdf = gdf_projected.cx[bbox.left:bbox.right, bbox.bottom:bbox.top]
        
        shapes = ((geom, value) for geom, value in zip(intersecting_gdf.geometry, intersecting_gdf[label_col]))
        try:
            full_mask = features.rasterize(
                shapes=shapes,
                out_shape=src.shape,
                transform=src.transform,
                fill=255,
                dtype=rasterio.uint8
            )
        except ValueError:
            full_mask = np.full(src.shape, 255, dtype=np.uint8)
            
        height, width = src.shape
        for row in range(0, height, PATCH_SIZE):
            for col in range(0, width, PATCH_SIZE):
                
                if row + PATCH_SIZE > height or col + PATCH_SIZE > width:
                    continue
                    
                window = Window(col, row, PATCH_SIZE, PATCH_SIZE)
                img_patch = src.read(window=window)
                mask_patch = full_mask[row:row+PATCH_SIZE, col:col+PATCH_SIZE]
                
                if np.any(mask_patch != 255):
                    patch_name = f"{tif_name.split('.')[0]}_{row}_{col}.tif"
                    out_img = f"{BASE_OUT}/{split}/images/{patch_name}"
                    out_mask = f"{BASE_OUT}/{split}/masks/{patch_name}"
                    
                    # Save Image
                    out_meta = src.meta.copy()
                    out_meta.update({
                        "height": PATCH_SIZE,
                        "width": PATCH_SIZE,
                        "transform": src.window_transform(window)
                    })
                    with rasterio.open(out_img, "w", **out_meta) as dest:
                        dest.write(img_patch)
                        
                    # Save Mask
                    mask_meta = out_meta.copy()
                    mask_meta.update({"count": 1, "dtype": "uint8"})
                    with rasterio.open(out_mask, "w", **mask_meta) as dest:
                        dest.write(mask_patch, 1)
                        
                    patch_counts[split] += 1

print("\n Processing Complete!")
print(f"Generated Patches -> Train: {patch_counts['train']}, Val: {patch_counts['val']}, Test: {patch_counts['test']}")

Processing 100 geographic tiles into 1024x1024 patches...
✅ Found label column: 'label_id'


Processing Tiles: 100%|███████████████████████| 100/100 [06:26<00:00,  3.86s/it]


 Processing Complete!
Generated Patches -> Train: 3782, Val: 330, Test: 324
